In [1]:
import pandas as pd

df = pd.read_csv('healthcare_dataset.csv')

print(df.shape)          # how many rows and columns
print(df.head())         # first 5 rows
print(df.dtypes)         # data types of each column
print(df.isnull().sum()) # any missing values?

(55500, 15)
            Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31   
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22   
3   andrEw waTtS   28  Female         O+          Diabetes        2020-11-18   
4  adrIENNE bEll   43  Female        AB+            Cancer        2022-09-19   

             Doctor                    Hospital Insurance Provider  \
0     Matthew Smith             Sons and Miller         Blue Cross   
1   Samantha Davies                     Kim Inc           Medicare   
2  Tiffany Mitchell                    Cook PLC              Aetna   
3       Kevin Wells  Hernandez Rogers and Vang,           Medicare   
4    Kathleen Hanna                 White-White              Aetna   

   Billing Amount  Room Number Admission Type Discharge Date   Medication  \
0    1885

In [2]:
# Fix messy name capitalization
df['Name'] = df['Name'].str.title()

# Fix date columns - use mixed format to handle inconsistencies
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], format='mixed', dayfirst=True)
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], format='mixed', dayfirst=True)

# New column: Length of Stay
df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

# New column: Age Groups
df['Age Group'] = pd.cut(df['Age'],
                          bins=[0, 18, 35, 55, 75, 100],
                          labels=['0-18', '19-35', '36-55', '56-75', '75+'])

# Check for any negative stay durations
print("Negative stays:", len(df[df['Length of Stay'] < 0]))

# Quick stats on two most important numeric columns
print("\nBilling Amount Stats:")
print(df['Billing Amount'].describe().round(2))

print("\nLength of Stay Stats:")
print(df['Length of Stay'].describe().round(2))

# Save cleaned file
df.to_csv('healthcare_cleaned.csv', index=False)
print("\nDone! Cleaned file saved.")

Negative stays: 0

Billing Amount Stats:
count    55500.00
mean     25539.32
std      14211.45
min      -2008.49
25%      13241.22
50%      25538.07
75%      37820.51
max      52764.28
Name: Billing Amount, dtype: float64

Length of Stay Stats:
count    55500.00
mean        15.51
std          8.66
min          1.00
25%          8.00
50%         15.00
75%         23.00
max         30.00
Name: Length of Stay, dtype: float64

Done! Cleaned file saved.


In [4]:
# Flag negative billing as anomalies
negative_billing = df[df['Billing Amount'] < 0]
print(f"Records with negative billing: {len(negative_billing)}")
print(negative_billing[['Name', 'Medical Condition', 'Billing Amount', 'Admission Type']].head(10))

# Distribution of key categorical columns
print("\nMedical Conditions:")
print(df['Medical Condition'].value_counts())

print("\nAdmission Types:")
print(df['Admission Type'].value_counts())

print("\nInsurance Providers:")
print(df['Insurance Provider'].value_counts())

print("\nTest Results:")
print(df['Test Results'].value_counts())

Records with negative billing: 108
                          Name Medical Condition  Billing Amount  \
132            Ashley Erickson            Cancer     -502.507813   
799          Christopher Weiss            Asthma    -1018.245371   
1018             Ashley Warner      Hypertension     -306.364925   
1421              Jay Galloway            Asthma     -109.097122   
2103         Joshua Williamson          Diabetes     -576.727907   
2696             Scott Vazquez          Diabetes     -135.986000   
2855            Carol Anderson      Hypertension     -370.983674   
3772  Mr. Christopher Alvarado           Obesity    -1310.272895   
5445            Alexandra Khan         Arthritis     -692.408820   
5708                Joseph Cox          Diabetes     -353.865186   

     Admission Type  
132          Urgent  
799        Elective  
1018       Elective  
1421      Emergency  
2103         Urgent  
2696       Elective  
2855       Elective  
3772       Elective  
5445       Electiv

In [3]:
df['Billing Flag'] = df['Billing Amount'].apply(
    lambda x: 'Anomaly' if x < 0 else 'Normal'
)

print(f"Clean records: {len(df[df['Billing Flag'] == 'Normal'])}")
print(f"Anomaly records: {len(df[df['Billing Flag'] == 'Anomaly'])}")

# Save final cleaned file with the flag column
df.to_csv('healthcare_cleaned.csv', index=False)
print("Updated cleaned file saved!")

Clean records: 55392
Anomaly records: 108
Updated cleaned file saved!
